# YOLO11s + LoRA v3 正式训练

这是与 v2 和主线模型完全隔离的实验。v3 固定使用 640 输入和 seed=42，采用 r=16、较低学习率和提前关闭 mosaic 的方案。正式输出写入 `models/trained/notebook/yolo11s_lora_formal_v3/`，不会覆盖 v2。

In [1]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'training').is_dir():
            return candidate
    raise FileNotFoundError('找不到项目根目录，请从项目内启动 Notebook')

PROJECT_ROOT = find_project_root(Path.cwd())
YOLO_MASTER_ROOT = PROJECT_ROOT / 'vendor/YOLO-Master'
if not (YOLO_MASTER_ROOT / 'ultralytics').is_dir():
    raise FileNotFoundError(f'找不到项目内 YOLO-Master: {YOLO_MASTER_ROOT}')

# PyCharm 有时会复用先前的 YOLO26 Kernel。无论当前解释器名称是什么，
# 都强制让本 Notebook 优先加载项目内支持 LoRA 的 YOLO-Master。
if str(YOLO_MASTER_ROOT) in sys.path:
    sys.path.remove(str(YOLO_MASTER_ROOT))
sys.path.insert(0, str(YOLO_MASTER_ROOT))
loaded_ultralytics = sys.modules.get('ultralytics')
if loaded_ultralytics is not None and 'YOLO-Master' not in str(getattr(loaded_ultralytics, '__file__', '')):
    for module_name in list(sys.modules):
        if module_name == 'ultralytics' or module_name.startswith('ultralytics.'):
            del sys.modules[module_name]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('项目根目录:', PROJECT_ROOT)
print('已优先绑定 YOLO-Master:', YOLO_MASTER_ROOT)

项目根目录: D:\dev\projects\yolo_train
已优先绑定 YOLO-Master: D:\dev\projects\yolo_train\vendor\YOLO-Master


In [2]:
import torch
import ultralytics
from ultralytics.cfg import DEFAULT_CFG_DICT

required = ('lora_r', 'lora_alpha', 'lora_dropout', 'lora_gradient_checkpointing')
missing = [key for key in required if key not in DEFAULT_CFG_DICT]
print('Python:', sys.version.split()[0])
print('Ultralytics / YOLO-Master:', ultralytics.__version__, ultralytics.__file__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if '.venv-yolo26' in str(Path(ultralytics.__file__).resolve()):
    raise RuntimeError('当前 Kernel 仍是 YOLO26 环境，请选择 Python (yolo) 后重启 Kernel')
if missing:
    raise RuntimeError(f'当前 Kernel 不是支持 LoRA 的 YOLO-Master 环境，缺少: {missing}')
if not torch.cuda.is_available():
    raise RuntimeError('正式训练要求 CUDA')

Python: 3.13.9
Ultralytics / YOLO-Master: 8.3.240 D:\dev\projects\yolo_train\vendor\YOLO-Master\ultralytics\__init__.py
CUDA: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


## 配方检查

默认 `RUN_FORMAL=True` 会执行最多 100 epochs 的正式训练。第一次建议先改为 `False` 跑 3 epochs 冒烟验证。两个模式使用不同目录。

In [3]:
from training.train_lora_v3 import FORMAL_NAME, SMOKE_NAME, v3_recipe

RUN_FORMAL = True
RESUME = False
EXPORT_ONNX = True
BATCH = 8

recipe = v3_recipe(smoke=not RUN_FORMAL)
print('运行类型:', '正式 v3' if RUN_FORMAL else '3 epochs smoke')
print('运行名称:', FORMAL_NAME if RUN_FORMAL else SMOKE_NAME)
recipe

D:\dev\runtimes\conda\envs\yolo\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


运行类型: 正式 v3
运行名称: yolo11s_lora_formal_v3


{'data': 'D:\\dev\\projects\\yolo_train\\data\\processed\\data.yaml',
 'epochs': 100,
 'imgsz': 640,
 'batch': 8,
 'device': 0,
 'workers': 0,
 'optimizer': 'AdamW',
 'lr0': 0.0003,
 'lrf': 0.05,
 'weight_decay': 0.0005,
 'patience': 30,
 'seed': 42,
 'deterministic': True,
 'amp': False,
 'pretrained': True,
 'cos_lr': True,
 'warmup_epochs': 3,
 'warmup_bias_lr': 0.01,
 'mosaic': 0.5,
 'close_mosaic': 40,
 'save_period': 10,
 'lora_r': 16,
 'lora_alpha': 16,
 'lora_dropout': 0.02,
 'lora_freeze_bn': True,
 'lora_use_rslora': False,
 'lora_init_lora_weights': 'gaussian',
 'lora_lr_mult': 0.5,
 'lora_alpha_warmup': 3,
 'lora_layer_decay': 1.0,
 'lora_ortho_weight': 0.0,
 'lora_dropout_end': 0.02,
 'lora_dropout_start_ratio': 1.0,
 'lora_gradient_checkpointing': False,
 'lora_include_head': False,
 'plots': True,
 'save': True,
 'verbose': True}

In [4]:
from training.train_lora_v3 import run_v3

result = run_v3(
    smoke=not RUN_FORMAL,
    resume=RESUME,
    export_onnx=EXPORT_ONNX,
    batch=BATCH,
)
result

New https://pypi.org/project/ultralytics/8.4.101 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.240  Python-3.13.9 torch-2.13.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=False, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=40, cls=0.5, cluster=False, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=D:\dev\projects\yolo_train\data\processed\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, iou_type=iou, keras=False, kobj=1.0, line_width=None, lora_adapter_dir=lora_adapter, lora_allow_depthwise=False, lora_allow_rtdetr_dora=False, lo

{'training_line': 'notebook_yolo11s_lora_v3_formal',
 'mainline_isolated': True,
 'automatically_deployed': False,
 'best_pt': 'D:\\dev\\projects\\yolo_train\\models\\trained\\notebook\\yolo11s_lora_formal_v3\\weights\\best.pt',
 'adapter_dir': 'D:\\dev\\projects\\yolo_train\\models\\trained\\notebook\\yolo11s_lora_formal_v3\\lora_adapter',
 'metrics_json': 'D:\\dev\\projects\\yolo_train\\models\\metrics\\notebook_lora_v3_formal_internal_metrics.json',
 'onnx': 'D:\\dev\\projects\\yolo_train\\models\\trained\\notebook\\yolo11s_lora_formal_v3\\weights\\best.onnx',
 'v3_minus_v2': {'precision': 0.009805405000837486,
  'recall': 0.06391859119786314,
  'map50': 0.029776150308653104,
  'map50_95': 0.048796611948439206},
 'metrics': {'source': 'yolo11s_lora_v3_formal',
  'model_path': 'D:\\dev\\projects\\yolo_train\\models\\trained\\notebook\\yolo11s_lora_formal_v3\\weights\\best.pt',
  'dataset': 'D:\\dev\\projects\\yolo_train\\data\\processed\\data.yaml',
  'split': 'test',
  'precision': 

## 训练趋势与 v2 对比

本单元格读取 v3 的 `results.csv`，不依赖当前 Kernel 中仍保留训练模型对象。

In [5]:
import json
import pandas as pd
import matplotlib.pyplot as plt

run_dir = PROJECT_ROOT / ('models/trained/notebook/yolo11s_lora_formal_v3' if RUN_FORMAL else 'models/trained/notebook_smoke/yolo11s_lora_v3_smoke')
df = pd.read_csv(run_dir / 'results.csv')
df.columns = [column.strip() for column in df.columns]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(df.index + 1, df['metrics/mAP50(B)'], label='mAP50')
axes[0].plot(df.index + 1, df['metrics/mAP50-95(B)'], label='mAP50-95')
axes[0].set(title='YOLO11s LoRA v3 validation metrics', xlabel='Epoch', ylabel='Metric')
axes[0].grid(alpha=.25); axes[0].legend()
for key in ('train/box_loss', 'train/cls_loss', 'train/dfl_loss'):
    axes[1].plot(df.index + 1, df[key], label=key.replace('train/', ''))
axes[1].set(title='Training losses', xlabel='Epoch', ylabel='Loss')
axes[1].grid(alpha=.25); axes[1].legend()
plt.tight_layout(); plt.show()

if RUN_FORMAL:
    comparison_path = PROJECT_ROOT / 'models/metrics/notebook_lora_v3_vs_v2.json'
    if comparison_path.is_file():
        comparison = json.loads(comparison_path.read_text(encoding='utf-8'))
        print('v3 - v2:', comparison['v3_minus_v2'])

<Figure size 1400x500 with 2 Axes>

v3 - v2: {'precision': 0.009805405000837486, 'recall': 0.06391859119786314, 'map50': 0.029776150308653104, 'map50_95': 0.048796611948439206}


## 当前模型训练与测试报告（可视化）

运行下面的 Cell 会汇总训练过程、固定 test 集核心指标、各类别 mAP50-95、混淆矩阵和预测样例，并将报告图片保存到当前模型目录的 `report/` 文件夹。

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import Markdown, display

# 自动选择本 Notebook 对应的正式或 smoke 训练产物。
report_run_dir = PROJECT_ROOT / ('models/trained/notebook/yolo11s_lora_formal_v3' if RUN_FORMAL else 'models/trained/notebook_smoke/yolo11s_lora_v3_smoke')
metric_path = PROJECT_ROOT / ('models/metrics/notebook_lora_v3_formal_internal_metrics.json' if RUN_FORMAL else 'models/metrics/notebook_lora_v3_smoke_internal_metrics.json')
evaluation_dir = PROJECT_ROOT / ('models/metrics/evaluation_runs/yolo11s_lora_v3_formal_internal' if RUN_FORMAL else 'models/metrics/evaluation_runs/yolo11s_lora_v3_smoke_internal')
report_dir = report_run_dir / 'report'
report_dir.mkdir(parents=True, exist_ok=True)

for required in (report_run_dir / 'results.csv', metric_path):
    if not required.is_file():
        raise FileNotFoundError(f'缺少报告输入文件：{required}')

history = pd.read_csv(report_run_dir / 'results.csv')
history.columns = [column.strip() for column in history.columns]
metrics = json.loads(metric_path.read_text(encoding='utf-8'))
epochs = np.arange(1, len(history) + 1)
best_epoch = int(history['metrics/mAP50-95(B)'].idxmax() + 1)
best_val_map = float(history['metrics/mAP50-95(B)'].max())

plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# 1. 训练集与验证集损失。
loss_pairs = (
    ('box', 'train/box_loss', 'val/box_loss', '#2E74B5'),
    ('cls', 'train/cls_loss', 'val/cls_loss', '#C58A1C'),
    ('dfl', 'train/dfl_loss', 'val/dfl_loss', '#1F6B45'),
)
for name, train_key, val_key, color in loss_pairs:
    axes[0, 0].plot(epochs, history[train_key], color=color, label=f'train {name}')
    axes[0, 0].plot(epochs, history[val_key], color=color, linestyle='--', alpha=.75, label=f'val {name}')
axes[0, 0].set(title='Training and validation losses', xlabel='Epoch', ylabel='Loss')
axes[0, 0].legend(ncol=2, fontsize=9)

# 2. 每轮验证指标。
curve_specs = (
    ('Precision', 'metrics/precision(B)', '#2E74B5'),
    ('Recall', 'metrics/recall(B)', '#D95F59'),
    ('mAP50', 'metrics/mAP50(B)', '#1F6B45'),
    ('mAP50-95', 'metrics/mAP50-95(B)', '#7A5195'),
)
for name, key, color in curve_specs:
    axes[0, 1].plot(epochs, history[key], color=color, label=name)
axes[0, 1].axvline(best_epoch, color='#111827', linestyle=':', label=f'best epoch {best_epoch}')
axes[0, 1].set(title='Validation metrics by epoch', xlabel='Epoch', ylabel='Metric', ylim=(0, 1))
axes[0, 1].legend(fontsize=9)

# 3. 固定 test 集核心指标。
metric_names = ['Precision', 'Recall', 'mAP50', 'mAP50-95']
metric_values = [metrics['precision'], metrics['recall'], metrics['map50'], metrics['map50_95']]
bars = axes[1, 0].bar(metric_names, metric_values, color=['#2E74B5', '#D95F59', '#1F6B45', '#7A5195'])
axes[1, 0].set(title='Fixed test-set metrics', ylabel='Score', ylim=(0, 1))
axes[1, 0].bar_label(bars, labels=[f'{value:.1%}' for value in metric_values], padding=3, fontsize=10)

# 4. 固定 test 集各类别定位指标。
class_metrics = metrics.get('per_class_map50_95', {})
class_names = list(class_metrics)
class_values = [class_metrics[name] for name in class_names]
class_bars = axes[1, 1].bar(class_names, class_values, color=['#4C78A8', '#F2CF5B', '#E45756'][:len(class_names)])
axes[1, 1].set(title='Per-class mAP50-95 on fixed test set', ylabel='mAP50-95', ylim=(0, 1))
axes[1, 1].bar_label(class_bars, labels=[f'{value:.1%}' for value in class_values], padding=3, fontsize=10)

for axis in axes.flat:
    axis.grid(alpha=.22)
fig.suptitle('YOLO11s LoRA v3 - Model Training Report', fontsize=18, fontweight='bold')
fig.tight_layout(rect=(0, 0, 1, .97))
training_report_path = report_dir / 'training_and_metrics_report.png'
fig.savefig(training_report_path, dpi=170, bbox_inches='tight')
plt.show()

# 固定 test 集的混淆矩阵与预测样例。Ultralytics 即使 split=test 也使用 val_batch 命名。
visual_paths = [
    ('Normalized confusion matrix', evaluation_dir / 'confusion_matrix_normalized.png'),
    ('Test predictions - batch 0', evaluation_dir / 'val_batch0_pred.jpg'),
    ('Test predictions - batch 1', evaluation_dir / 'val_batch1_pred.jpg'),
    ('Test predictions - batch 2', evaluation_dir / 'val_batch2_pred.jpg'),
]
available_visuals = [(title, path) for title, path in visual_paths if path.is_file()]
if available_visuals:
    fig2, visual_axes = plt.subplots(2, 2, figsize=(17, 12))
    for axis, (title, path) in zip(visual_axes.flat, available_visuals):
        with Image.open(path) as image:
            axis.imshow(np.asarray(image))
        axis.set_title(title, fontsize=12)
        axis.axis('off')
    for axis in visual_axes.flat[len(available_visuals):]:
        axis.axis('off')
    fig2.suptitle('Fixed Test Set - Confusion Matrix and Prediction Samples', fontsize=17, fontweight='bold')
    fig2.tight_layout(rect=(0, 0, 1, .97))
    test_report_path = report_dir / 'fixed_test_visual_report.png'
    fig2.savefig(test_report_path, dpi=170, bbox_inches='tight')
    plt.show()
else:
    test_report_path = None
    print(f'未找到固定 test 集可视化文件：{evaluation_dir}')

speed = metrics.get('speed_ms', {})
summary = f"""
### 模型报告摘要

| 项目 | 结果 |
|---|---:|
| 完成训练轮次 | {len(history)} |
| 最佳验证轮次 | {best_epoch} |
| 最佳验证 mAP50-95 | {best_val_map:.2%} |
| 固定 test Precision | {metrics['precision']:.2%} |
| 固定 test Recall | {metrics['recall']:.2%} |
| 固定 test mAP50 | {metrics['map50']:.2%} |
| 固定 test mAP50-95 | {metrics['map50_95']:.2%} |
| 单张推理时间 | {speed.get('inference', float('nan')):.2f} ms |
| 模型大小 | {metrics.get('model_size_mb', float('nan')):.2f} MB |

- 模型权重：`{metrics.get('model_path', report_run_dir / 'weights/best.pt')}`
- 训练与指标报告：`{training_report_path}`
- 测试可视化报告：`{test_report_path or '未生成'}`
"""
display(Markdown(summary))

NameError: name 'PROJECT_ROOT' is not defined